# Guide 4 — ArUco Markers (the four corner stickers)

> **PYNQ Bootcamp guide.** This notebook takes one piece of the big competition program and explains it in small steps. Almost all of the code here is the *real* code that runs during a match — we've just split it up and added plain-English notes so it's easy to follow. (The one exception is the *Matching Strategy* guide, where the game plan is written as pseudocode for you to think through.)

## What is this notebook about?

ArUco markers are little black-and-white square patterns — like simple QR codes.
Four of them sit at the corners of the board. When the camera spots them, the
program instantly knows where the board's corners are, at any angle.

This guide is all real code. It finds the markers and uses them to line up the
board (that's what Guide 3 needed).


### How this guide fits in

**Depends on:** Guides 1 and 3 (grid math). **Used by:** Guide 5, and the main turn loop.

*New here? Read **Guide 0 — How Everything Connects** first for the big picture.*


### Which markers are which

These settings say what kind of markers we use and which four ID numbers are the
corners. Notice the corner IDs are `3, 4, 6, 5` — those numbers are their *jobs*
(top-left, top-right, bottom-right, bottom-left).


In [ ]:
# Which ArUco dictionary the printed markers use, and which four IDs mark the
# arena's physical corners (aruco_border mode) vs. individual cards (aruco_per_card).
ARUCO_DICTIONARY_NAME = 'DICT_4X4_50'
# Keep these synchronized with the Blue notebook: semantic board-corner IDs,
# not image-space positions, define orientation for every camera.
BORDER_MARKER_TL = 3
BORDER_MARKER_TR = 4
BORDER_MARKER_BR = 6
BORDER_MARKER_BL = 5
BOARD_MARKER_CENTERS = {}
BOARD_CAMERA_ROTATION_DEGREES = None

# Fixed floor for crop-space detections -- distinct from the fallback ladder
# below, which only applies to detect_position's own retry loop.
GRID_CROP_SIZE = (416, 416)

# Lower thresholds detect_position retries at, in order, before giving up.
YOLO_FALLBACK_SCORE_THRESHOLDS = [0.15, 0.1, 0.05]

# How many fresh photos detect_position takes per threshold before moving to
# the next (lower) one -- a bit of multi-photo voting cushions against one
# unlucky frame (motion blur, a hand briefly in view) without adding much
# latency, since the fallback ladder only kicks in on genuine misses.
DETECT_PHOTO_COUNT = 2
DETECT_SETTLE_SECONDS = 0.3
DETECT_FLUSH_FRAMES = 8

# Only used by the debug overlay (show_detection_frame) -- how many of the
# top-scoring records to draw boxes/labels for.
MAX_OBJECTS = 2
TURN_DEBUG_SHOW_LABELS = True
TURN_DEBUG_SHOW_ALL_DETECTIONS = False

_APPROACH_SETTINGS = {
    'yolo_full_frame': ('full_frame', 'yolo_center'),
    'yolo_grid_crops': ('grid',       'yolo_center'),
    'aruco_border':    ('grid',       'yolo_center'),
    'aruco_per_card':  ('full_frame', 'aruco_id'),
}

### Picking the detection style

The program supports four ways to find cards; this match uses `aruco_border`. These
helper functions just switch between the styles. Run as-is.


In [ ]:
def set_detection_mode(mode):
    global DETECTION_MODE
    if mode not in ('grid', 'full_frame'):
        raise ValueError("mode must be 'grid' or 'full_frame'")
    DETECTION_MODE = mode
    return DETECTION_MODE


def set_card_position_mode(mode):
    global CARD_POSITION_MODE
    if mode not in ('yolo_center', 'aruco_id'):
        raise ValueError("mode must be 'yolo_center' or 'aruco_id'")
    CARD_POSITION_MODE = mode
    return CARD_POSITION_MODE


def set_detection_approach(approach):
    global DETECTION_APPROACH
    if approach not in _APPROACH_SETTINGS:
        raise ValueError(f"Unknown approach: {approach!r}. Choose from: {list(_APPROACH_SETTINGS)}")
    DETECTION_APPROACH = approach
    det_mode, pos_mode = _APPROACH_SETTINGS[approach]
    set_detection_mode(det_mode)
    set_card_position_mode(pos_mode)
    print(f'[detection] approach set to: {approach}')

### Finding the markers in a picture

`detect_aruco_corners_and_ids` looks at a picture and returns every marker it found
and its ID number. This is the real "camera, do you see any markers?" function.


In [ ]:
def aruco_dictionary():
    if not hasattr(cv2, 'aruco'):
        raise RuntimeError('OpenCV ArUco support is not available. Install opencv-contrib-python or use an OpenCV build with cv2.aruco.')
    dictionary_id = getattr(cv2.aruco, ARUCO_DICTIONARY_NAME, None)
    if dictionary_id is None:
        raise ValueError(f'Unknown ArUco dictionary: {ARUCO_DICTIONARY_NAME}')
    return cv2.aruco.getPredefinedDictionary(dictionary_id)


def aruco_detector_parameters():
    if hasattr(cv2.aruco, 'DetectorParameters'):
        return cv2.aruco.DetectorParameters()
    return cv2.aruco.DetectorParameters_create()


def detect_aruco_corners_and_ids(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    dictionary = aruco_dictionary()
    parameters = aruco_detector_parameters()
    if hasattr(cv2.aruco, 'ArucoDetector'):
        detector = cv2.aruco.ArucoDetector(dictionary, parameters)
        corners, ids, rejected = detector.detectMarkers(gray)
    else:
        corners, ids, rejected = cv2.aruco.detectMarkers(gray, dictionary, parameters=parameters)
    if ids is None:
        return [], np.array([], dtype=np.int32)
    return corners, ids.flatten().astype(np.int32)

### Using the corners to line up the board

`calibrate_from_border_markers` is the important one: it takes the four corner
markers and figures out exactly where the board is, then hands that to the grid
math from Guide 3. Because it sorts markers by their job, the board always comes out
right-side up.


In [ ]:
def calibrate_from_border_markers(frame):
    """Update the board homography from semantic TL/TR/BR/BL marker IDs.

    Marker roles are fixed to the physical board. Their camera-image locations
    may rotate freely, including a 180-degree upside-down camera; mapping the
    role-ordered source points to canonical corners keeps A1 at top-left.
    """
    global BOARD_CORNERS, BOARD_MARKER_CENTERS, BOARD_CAMERA_ROTATION_DEGREES
    corners, ids = detect_aruco_corners_and_ids(frame)
    marker_roles = {
        BORDER_MARKER_TL: ('TL', 0),
        BORDER_MARKER_TR: ('TR', 1),
        BORDER_MARKER_BR: ('BR', 2),
        BORDER_MARKER_BL: ('BL', 3),
    }
    found = {}
    centers_by_role = {}
    for marker_corners, marker_id in zip(corners, ids):
        role_info = marker_roles.get(int(marker_id))
        if role_info is None:
            continue
        role, corner_index = role_info
        center = np.mean(marker_corners.reshape(4, 2), axis=0).astype(np.float32)
        found[corner_index] = center
        centers_by_role[role] = center
    if len(found) == 4:
        candidate = np.float32([found[0], found[1], found[2], found[3]])
        if abs(float(cv2.contourArea(candidate))) < 100.0:
            raise RuntimeError('Border marker centers form a degenerate board quadrilateral.')
        BOARD_CORNERS = candidate
        BOARD_MARKER_CENTERS = centers_by_role
        top_edge = found[1] - found[0]
        BOARD_CAMERA_ROTATION_DEGREES = float(np.degrees(np.arctan2(top_edge[1], top_edge[0])))
    return len(found)

### Turning markers into card records

The rest of this section turns detections into tidy "record" cards the program
passes around, and picks the right detection style each turn. It's longer, but it's
all bookkeeping — read the comments and run it.


In [ ]:
def aruco_records_for_frame(frame, detection_mode):
    """Detect ArUco markers on the full frame, mapping each to a grid cell.
    Only card-range IDs (0..GRID_ROWS*GRID_COLS-1) are returned; configured
    border marker IDs never appear as false card detections."""
    corners, ids = detect_aruco_corners_and_ids(frame)
    records = []
    card_id_limit = GRID_ROWS * GRID_COLS
    for marker_corners, marker_id in zip(corners, ids):
        if int(marker_id) >= card_id_limit:
            continue
        points = marker_corners.reshape(4, 2).astype(np.float32)
        center = tuple(np.mean(points, axis=0).astype(float))
        grid_info = aruco_grid_position(marker_id, center, frame.shape)
        x_min, y_min = np.min(points, axis=0)
        x_max, y_max = np.max(points, axis=0)
        records.append({
            'description': f'aruco_{int(marker_id)}', 'score': 1.0, 'class_index': int(marker_id),
            'aruco_id': int(marker_id), 'row': grid_info['row'], 'col': grid_info['col'],
            'cell': grid_info['cell'], 'center': grid_info['center'],
            'corners': [[float(x), float(y)] for x, y in points],
            'box': [float(y_min), float(x_min), float(y_max), float(x_max)],
            'crop_box': [float(y_min), float(x_min), float(y_max), float(x_max)],
            'detection_mode': detection_mode, 'detector': 'aruco',
        })
    return sorted(records, key=lambda record: (record['row'], record['col'], record['aruco_id']))


def aruco_records_for_grid_cell(frame, row, col):
    """Crop one grid cell, detect ArUco markers in that stretched crop, and map them back to the frame."""
    crop, src_quad, dst_quad = crop_grid_cell(frame, row, col)
    corners, ids = detect_aruco_corners_and_ids(crop)
    records = []
    for marker_corners, marker_id in zip(corners, ids):
        crop_points = marker_corners.reshape(4, 2).astype(np.float32)
        frame_points = crop_points_to_frame(crop_points, src_quad, dst_quad)
        center = tuple(np.mean(frame_points, axis=0).astype(float))
        grid_info = aruco_grid_position(marker_id, center, frame.shape)
        crop_x_min, crop_y_min = np.min(crop_points, axis=0)
        crop_x_max, crop_y_max = np.max(crop_points, axis=0)
        frame_x_min, frame_y_min = np.min(frame_points, axis=0)
        frame_x_max, frame_y_max = np.max(frame_points, axis=0)
        records.append({
            'description': f'aruco_{int(marker_id)}', 'score': 1.0, 'class_index': int(marker_id),
            'aruco_id': int(marker_id), 'row': grid_info['row'], 'col': grid_info['col'],
            'cell': grid_info['cell'], 'center': grid_info['center'],
            'corners': [[float(x), float(y)] for x, y in frame_points],
            'box': [float(frame_y_min), float(frame_x_min), float(frame_y_max), float(frame_x_max)],
            'crop_box': [float(crop_y_min), float(crop_x_min), float(crop_y_max), float(crop_x_max)],
            'detection_mode': 'grid', 'detector': 'aruco',
        })
    return records


def yolo_record_from_box(frame_shape, box, score, class_index, detection_mode, crop_box=None):
    y_min, x_min, y_max, x_max = map(float, box)
    center = ((x_min + x_max) / 2.0, (y_min + y_max) / 2.0)
    grid_info = grid_position_for_point(center, frame_shape)
    return {
        'description': class_names[int(class_index)], 'score': float(score), 'class_index': int(class_index),
        'row': grid_info['row'], 'col': grid_info['col'], 'cell': grid_info['cell'], 'center': grid_info['center'],
        'box': [y_min, x_min, y_max, x_max],
        'crop_box': [float(value) for value in (crop_box if crop_box is not None else box)],
        'detection_mode': detection_mode, 'detector': 'yolo',
    }


def best_detection_for_grid_cell(frame, row, col, score_thresh=None):
    """Run YOLO on one perspective-corrected grid cell and map the best box back to the frame."""
    crop, src_quad, dst_quad = crop_grid_cell(frame, row, col)
    boxes, scores, classes = run(crop, score_thresh=score_thresh)
    # This is the exact 416x416 image passed to YOLO, with crop-space boxes.
    # The widget is created in the Camera cell and displayed only in the dashboard.
    if 'processed_crop_image_widget' in globals():
        show_processed_grid_crop(crop, boxes, scores, classes, pos_name(row, col))
    if not scores.any():
        return None
    best_idx = int(np.argmax(scores))
    crop_box = boxes[best_idx]
    frame_box = crop_box_to_frame_box(crop_box, src_quad, dst_quad, frame.shape)
    record = yolo_record_from_box(frame.shape, frame_box, scores[best_idx], classes[best_idx], detection_mode='grid', crop_box=crop_box)
    record['row'] = int(row)
    record['col'] = int(col)
    record['cell'] = pos_name(row, col)
    record['center'] = crop_box_center_to_frame(crop_box, src_quad, dst_quad)
    return record


def yolo_records_for_grid(frame, score_thresh=None):
    records = []
    for row in range(GRID_ROWS):
        for col in range(GRID_COLS):
            record = best_detection_for_grid_cell(frame, row, col, score_thresh=score_thresh)
            if record is not None:
                records.append(record)
    return sorted(records, key=lambda record: record['score'], reverse=True)


def aruco_records_for_grid(frame):
    records = []
    for row in range(GRID_ROWS):
        for col in range(GRID_COLS):
            records.extend(aruco_records_for_grid_cell(frame, row, col))
    return sorted(records, key=lambda record: (record['row'], record['col'], record['aruco_id']))


def yolo_records_for_full_frame(frame, score_thresh=None):
    boxes, scores, classes = run(frame, score_thresh=score_thresh)
    if not scores.any():
        return []
    records = [
        yolo_record_from_box(frame.shape, box, score, class_index, detection_mode='full_frame')
        for box, score, class_index in zip(boxes, scores, classes)
    ]
    return sorted(records, key=lambda record: record['score'], reverse=True)


def record_selection_key(record):
    return (record.get('detector') == 'yolo', 'aruco_id' in record, record.get('score', 1.0))


def merge_aruco_with_yolo_records(yolo_records, aruco_records):
    """Attach nearest ArUco IDs to YOLO cards, optionally using marker IDs for stored positions."""
    merged_records = [dict(record) for record in yolo_records]
    used_aruco_indices = set()
    for yolo_record in merged_records:
        if not aruco_records:
            break
        yolo_center = np.array(yolo_record['center'], dtype=np.float32)
        candidate_indices = [i for i in range(len(aruco_records)) if i not in used_aruco_indices]
        if not candidate_indices:
            break
        best_index = min(
            candidate_indices,
            key=lambda index: np.linalg.norm(np.array(aruco_records[index]['center'], dtype=np.float32) - yolo_center),
        )
        aruco_record = aruco_records[best_index]
        aruco_center = tuple(map(float, aruco_record['center']))
        yolo_record['yolo_cell'] = yolo_record['cell']
        yolo_record['yolo_center'] = yolo_record['center']
        yolo_record['aruco_id'] = int(aruco_record['aruco_id'])
        yolo_record['aruco_score'] = float(aruco_record.get('score', 1.0))
        yolo_record['aruco_center'] = aruco_center
        yolo_record['corners'] = aruco_record['corners']
        if CARD_POSITION_MODE == 'aruco_id':
            grid_info = grid_position_for_aruco_id(aruco_record['aruco_id'], aruco_center)
            yolo_record['row'] = int(grid_info['row'])
            yolo_record['col'] = int(grid_info['col'])
            yolo_record['cell'] = grid_info['cell']
        used_aruco_indices.add(best_index)
    unmatched_aruco_records = [dict(r) for i, r in enumerate(aruco_records) if i not in used_aruco_indices]
    merged_records.extend(unmatched_aruco_records)
    return sorted(merged_records, key=record_selection_key, reverse=True)


def scan_grid_cells_for_objects(frame, score_thresh=None):
    """Crop/stretch each grid cell, then run YOLO and ArUco marker detection on each crop. ('yolo_grid_crops')"""
    yolo_records = yolo_records_for_grid(frame, score_thresh=score_thresh)
    aruco_records = aruco_records_for_grid(frame)
    return merge_aruco_with_yolo_records(yolo_records, aruco_records)


def full_frame_records(frame, score_thresh=None):
    """Run YOLO and ArUco marker detection on the full camera frame."""
    yolo_records = yolo_records_for_full_frame(frame, score_thresh=score_thresh)
    aruco_records = aruco_records_for_frame(frame, detection_mode='full_frame')
    return merge_aruco_with_yolo_records(yolo_records, aruco_records)


def records_for_current_mode(frame, score_thresh=None):
    """Detect face-up cards using the configured DETECTION_APPROACH -- the
    single entry point detect_position() calls below, regardless of which
    of the four approaches is active."""
    approach = globals().get('DETECTION_APPROACH', 'aruco_per_card')
    if approach == 'yolo_full_frame':
        return yolo_records_for_full_frame(frame, score_thresh=score_thresh)
    if approach == 'yolo_grid_crops':
        return scan_grid_cells_for_objects(frame, score_thresh=score_thresh)
    if approach == 'aruco_border':
        n = calibrate_from_border_markers(frame)
        if n < 4:
            print(f'[aruco_border] {n}/4 border markers visible -- using the last valid board calibration')
        return scan_grid_cells_for_objects(frame, score_thresh=score_thresh)
    if approach == 'aruco_per_card':
        yolo_records = yolo_records_for_full_frame(frame, score_thresh=score_thresh)
        aruco_records = aruco_records_for_frame(frame, detection_mode='full_frame')
        return merge_aruco_with_yolo_records(yolo_records, aruco_records)
    raise ValueError(f"Unknown DETECTION_APPROACH: {approach!r}. Choose from: {list(_APPROACH_SETTINGS)}")


def turn_score_thresholds(base_threshold=None, fallback_thresholds=None):
    """Base threshold followed by lower fallback thresholds, descending, deduplicated."""
    base_threshold = YOLO_SCORE_THRESHOLD if base_threshold is None else base_threshold
    if fallback_thresholds is None:
        fallback_thresholds = YOLO_FALLBACK_SCORE_THRESHOLDS
    thresholds = [float(base_threshold)]
    for threshold in sorted({float(v) for v in fallback_thresholds}, reverse=True):
        if threshold < base_threshold and not any(np.isclose(threshold, existing) for existing in thresholds):
            thresholds.append(threshold)
    return thresholds


def draw_detection_label(frame, label, x, y, color=(255, 255, 255)):
    text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
    y = max(y, text_size[1] + 8)
    cv2.rectangle(frame, (x, y - text_size[1] - 8), (x + text_size[0] + 8, y + 4), (0, 0, 0), -1)
    return cv2.putText(frame, label, (x + 4, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)


def record_color(record):
    if not len(colors):
        return (255, 255, 255)
    return colors[int(record.get('class_index', 0)) % len(colors)]


def draw_aruco_marker(frame, record, color):
    points = np.array(record['corners'], dtype=np.int32)
    cv2.polylines(frame, [points], True, color, 2)
    marker_center = record.get('aruco_center', record['center'])
    center_x, center_y = map(int, marker_center)
    cv2.circle(frame, (center_x, center_y), 4, color, -1)
    return frame


def marker_label(record):
    if 'aruco_id' in record:
        if record.get('detector') == 'yolo':
            return f"{record['description']} {record['score']:.2f} ID {record['aruco_id']} {record['cell']}"
        return f"ID {record['aruco_id']} {record['cell']}"
    return f"{record['description']} {record['score']:.2f} {record['cell']}"


def annotate_records_frame(frame, records, turn_label='Current detection'):
    annotated = draw_grid(frame)
    sorted_records = sorted(records, key=lambda record: record.get('score', 1.0), reverse=True)
    debug_records = sorted_records if TURN_DEBUG_SHOW_ALL_DETECTIONS else sorted_records[:MAX_OBJECTS]
    for object_number, record in enumerate(debug_records, start=1):
        y_min, x_min, y_max, x_max = map(int, record['box'])
        color = record_color(record)
        if record.get('detector') == 'yolo':
            annotated = cv2.rectangle(annotated, (x_min, y_min), (x_max, y_max), color, 2)
        if 'corners' in record:
            annotated = draw_aruco_marker(annotated, record, color)
        elif record.get('detector') != 'yolo':
            annotated = cv2.rectangle(annotated, (x_min, y_min), (x_max, y_max), color, 2)
        if TURN_DEBUG_SHOW_LABELS:
            label = f"{object_number}: {marker_label(record)}"
            annotated = draw_detection_label(annotated, label, x_min, y_min, color)
    marker_count = len([r for r in records if 'aruco_id' in r])
    yolo_count = len([r for r in records if r.get('detector') == 'yolo'])
    mode_word = 'grid mode' if DETECTION_MODE == 'grid' else 'full-frame mode'
    status = f"{turn_label}: {mode_word}, found {yolo_count} YOLO object(s), {marker_count} ArUco marker(s)"
    annotated = draw_detection_label(annotated, status, 20, 30, (255, 255, 255))
    if len(records) < MAX_OBJECTS:
        warning = 'Expected two revealed cards. Check YOLO threshold, ArUco dictionary, grid alignment, lighting, or visibility.'
        annotated = draw_detection_label(annotated, warning, 20, 65, (0, 255, 255))
    return annotated


def _set_image_widget(widget, frame):
    ok, encoded = cv2.imencode('.jpeg', frame)
    if not ok:
        raise RuntimeError('Could not encode debug image.')
    widget.value = encoded.tobytes()


def show_detection_frame(frame, records, turn_label='Current detection'):
    """Update the dashboard's board-space view without creating a new output."""
    annotated = annotate_records_frame(frame, records, turn_label=turn_label)
    if 'detection_image_widget' in globals():
        _set_image_widget(detection_image_widget, annotated)


def annotate_processed_grid_crop(crop, boxes, scores, classes, cell):
    """Draw crop-space YOLO results on the exact 416x416 input image."""
    annotated = crop.copy()
    for box, score, class_idx in zip(boxes, scores, classes):
        y_min, x_min, y_max, x_max = map(int, box)
        color = colors[int(class_idx) % len(colors)]
        cv2.rectangle(annotated, (x_min, y_min), (x_max, y_max), color, 2)
        label = f'{class_names[int(class_idx)]} {float(score):.2f}'
        annotated = draw_detection_label(annotated, label, x_min, max(18, y_min), color)
    summary = f'{cell} processed 416x416 crop: {len(boxes)} detection(s)'
    return draw_detection_label(annotated, summary, 8, 22, (255, 255, 255))


def show_processed_grid_crop(crop, boxes, scores, classes, cell):
    """Keep the dashboard fixed while replacing only its processed-crop image."""
    annotated = annotate_processed_grid_crop(crop, boxes, scores, classes, cell)
    if 'processed_crop_image_widget' in globals():
        _set_image_widget(processed_crop_image_widget, annotated)


def print_aruco_marker_report(records):
    marker_records = [r for r in records if 'aruco_id' in r]
    if not marker_records:
        print('No ArUco markers detected')
        return
    print('ArUco markers detected:')
    for record in sorted(marker_records, key=lambda item: (item['row'], item['col'], item['aruco_id'])):
        center_x, center_y = record.get('aruco_center', record['center'])
        print(f"  id={record['aruco_id']} cell={record['cell']} row={record['row'] + 1} col={record['col'] + 1} center=({center_x:.1f}, {center_y:.1f})")

# Fixed for this notebook -- runs once at import time so DETECTION_MODE/CARD_POSITION_MODE always match DETECTION_APPROACH, with no dropdown needed to trigger it.
set_detection_approach(DETECTION_APPROACH)

### Check yourself

1. Why are the corner markers labeled by *job* (top-left, etc.) instead of by
   where they appear in the picture?
2. What does `calibrate_from_border_markers` give the grid math from Guide 3?
